## Imports

In [67]:
# GIS and spatial data libraries
import rasterio
from rasterio.crs import CRS
from rasterio.transform import xy
from affine import Affine

# Data analysis libraries
import numpy as np

# Data access libraries
import json
from pathlib import Path
import pip
import platform
import sys

# Check for load errors
print(
    "All libraries imported successfully\n" 
    f"pip v.{pip.__version__}\n"
    f"Python v{platform.python_version()}"
)

All libraries imported successfully
pip v.25.2
Python v3.13.7


In [68]:
# Verify paths
print(
    f"Working directory: {Path.cwd()}\n" 
    f"Python executable: {sys.executable}\n"
    f"Datasets folder exists: {Path("Datasets").exists()}\n"
    f"Assets folder exists: {Path("Assets").exists()}\n"
)

Working directory: c:\Users\matil\Documents\GitHub\GIS-Project
Python executable: c:\Users\matil\AppData\Local\Programs\Python\Python313\python.exe
Datasets folder exists: True
Assets folder exists: True



## Band and Blender Exports

In [69]:
# Select clean data for export
with np.load(r"Data Export\export_inputs.npz") as data:
    elevation_crop   = data["elevation_crop"]
    slope_gdal_array = data["slope_gdal_array"]
    ndvi_crop        = [data["ndvi_2024"], data["ndvi_2025"], data["ndvi_2026"]]
    sentinel_transform = Affine(*data["sentinel_transform"])
    col0, row0 = int(data["col0"]), int(data["row0"])
    dw_crop = data["dw_crop"]
    sentinel_crs = CRS.from_wkt(str(data["sentinel_crs"]))

### Multi-band Export to Blender

In [70]:
# Shared validity mask
valid = (
    np.isfinite(elevation_crop)
    & np.isfinite(slope_gdal_array)
    & np.isfinite(ndvi_crop[0])
    & np.isfinite(ndvi_crop[1])
    & np.isfinite(ndvi_crop[2])
    & np.isfinite(dw_crop)
)

elev = np.where(valid, elevation_crop, np.nan)
slope = np.where(valid, slope_gdal_array, np.nan)
ndvi_24 = np.where(valid, ndvi_crop[0], np.nan)
ndvi_25 = np.where(valid, ndvi_crop[1], np.nan)
ndvi_26 = np.where(valid, ndvi_crop[2], np.nan)
dw = np.where(valid, dw_crop, np.nan)

# Select export bounds
rows, cols = np.indices(elev.shape)
rows, cols = rows.ravel(), cols.ravel()

# Transform pixels back into UTM values
crop_transform = sentinel_transform * Affine.translation(col0, row0)
x_utm, y_utm = rasterio.transform.xy(crop_transform, rows, cols, offset="center")
x_utm, y_utm = np.asarray(x_utm), np.asarray(y_utm)

# Set origin
origin_x, origin_y = crop_transform * (0, 0)
z_shift = 0.0     # Keep true elevations

# Shift points to new origin
blender_points = {
    "x_pos_bld":     x_utm - origin_x,
    "y_pos_bld":     y_utm - origin_y,
    "z_elev_bld":    elev[rows, cols] - z_shift,
    "slope_degrees": slope[rows, cols],
    "ndvi_2024":     ndvi_24[rows, cols],
    "ndvi_2025":     ndvi_25[rows, cols],
    "ndvi_2026":     ndvi_26[rows, cols],
    "dw_class":      dw[rows, cols],
}

In [71]:
# Check points
for name in ("x_pos_bld", "y_pos_bld", "z_elev_bld", "slope_degrees", "ndvi_2024", "ndvi_2025", "ndvi_2026", "dw_class"):
    arr = blender_points[name]
    print(f"{name}: min={np.nanmin(arr):.1f}  max={np.nanmax(arr):.1f}  span={np.nanmax(arr)-np.nanmin(arr):.1f}")
print(f"points: {len(rows)}")
print(f"Input dataset size: {elevation_crop.shape}")
print(f"Input dataset datapoint counts: {elevation_crop.shape[0]*elevation_crop.shape[1]}")

# Write points to CSV
np.savetxt(
    r"Data Export\terrain_points.csv",
    np.column_stack(list(blender_points.values())),
    delimiter=",",
    header=",".join(blender_points),
    comments="",
    fmt="%.3f",
)

x_pos_bld: min=5.0  max=10425.0  span=10420.0
y_pos_bld: min=-24855.0  max=-5.0  span=24850.0
z_elev_bld: min=-0.0  max=1492.6  span=1492.7
slope_degrees: min=0.0  max=73.4  span=73.4
ndvi_2024: min=-1.0  max=1.0  span=2.0
ndvi_2025: min=-1.0  max=1.0  span=2.0
ndvi_2026: min=-1.0  max=1.0  span=2.0
dw_class: min=0.0  max=8.0  span=8.0
points: 2592898
Input dataset size: (2486, 1043)
Input dataset datapoint counts: 2592898


### Multi-band Export to QGIS

In [72]:
# Store bands for export into dictionary
clean_export_datasets = {
    "NDVI_2024": ndvi_crop[0],
    "NDVI_2025": ndvi_crop[1],
    "NDVI_2026": ndvi_crop[2],
    "DEM": elevation_crop,
    "Slope": slope_gdal_array,
    "DW_v1": dw_crop
}

# Verify clean export bands have matching shapes
for name, array in clean_export_datasets.items():
    print(f"Key: {name}, Array Shape: {array.shape}")

# Select width and height of export datasets
export_array_width, export_array_height = clean_export_datasets["NDVI_2024"].shape[1], clean_export_datasets["NDVI_2024"].shape[0]

Key: NDVI_2024, Array Shape: (2486, 1043)
Key: NDVI_2025, Array Shape: (2486, 1043)
Key: NDVI_2026, Array Shape: (2486, 1043)
Key: DEM, Array Shape: (2486, 1043)
Key: Slope, Array Shape: (2486, 1043)
Key: DW_v1, Array Shape: (2486, 1043)


In [ ]:
# Specify export path and crop the grid transform to match the cropped datasets
output_path = r"Data Export\export_bands.tif"
crop_transform = sentinel_transform * Affine.translation(col0, row0)

# Export clean datasets to GeoTIFF
with rasterio.open(output_path, "w",
                   driver="GTiff",
                   height=export_array_height,
                   width=export_array_width,
                   count=len(clean_export_datasets),
                   dtype="float32",
                   crs=sentinel_crs,
                   transform=crop_transform,
                   nodata=np.nan) as dst:

    for band_number, (name, raster) in enumerate(clean_export_datasets.items(), start=1):
        dst.write(raster.astype("float32"), band_number)
        dst.set_band_description(band_number, name)